# Toronto Firearm Incident Analysis

**Course:** CCT416 · **Stack:** Python, Pandas, Matplotlib, DuckDB (SQL)

## Research question
Do Toronto neighbourhoods with the **most firearm-related incidents** also show the **most harm** (injuries + deaths), or does “high risk” change when we switch from counting events to measuring severity?

## Why this matters (interview framing)
Public safety dashboards often rank places by **volume**. Volume and **harm** can tell different stories. This project tests that claim with open data, then **re-checks the same rankings in SQL** so the result isn’t locked to one tool.

## Pipeline at a glance
1. Load Toronto Police Service open data
2. Clean & engineer a harm metric
3. Rank neighbourhoods three ways (Pandas)
4. Visualize & save charts
5. Reload cleaned rows into **DuckDB** and rebuild top-10 rankings with **SQL**
6. Compare methods and state limitations

## 0. Setup — libraries & chart folder

**Interview tip:** Import only what you use. `pathlib` + `makedirs` keeps chart outputs reproducible for a portfolio repo.

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

# Folder for portfolio-ready chart images (committed to GitHub)
CHARTS = Path("charts")
CHARTS.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 50)
print("Libraries loaded. Charts will save to:", CHARTS.resolve())

## 1. Load the raw dataset

**Source:** [Toronto Police Service — Shooting and Firearm Discharges Open Data](https://data.torontopolice.on.ca/datasets/TorontoPS::shooting-and-firearm-discharges-open-data/about)

`pd.read_csv` reads the file into a DataFrame (rows = incidents, columns = fields).
`.head()` is a quick sanity check that columns look right before cleaning.

In [ ]:
df = pd.read_csv("shooting_firearm_discharges.csv")
print("Raw shape (rows, columns):", df.shape)
df.head()

## 2. Explore structure, types, and missingness

Before ranking anything, confirm:
- how many rows/columns you have
- which fields exist
- whether nulls will break aggregations

**Interview tip:** Saying “I profiled nulls and dtypes before aggregation” shows you don’t jump straight to charts.

In [ ]:
print("Column names:")
print(df.columns.tolist())
print()
df.info()
print()
print("Missing values per column:")
print(df.isnull().sum())

## 3. Select fields, coerce types, engineer harm, drop unclear geography

**Columns we keep**
- `OCC_YEAR` — when the incident occurred
- `NEIGHBOURHOOD_158` — City of Toronto 158-neighbourhood geography
- `EVENT_TYPE` — Shooting vs Firearm Discharge
- `INJURIES`, `DEATH` — severity inputs

**Cleaning choices (be ready to defend these)**
1. Coerce year / injuries / deaths to numeric (`errors="coerce"` turns bad values into NaN).
2. Drop rows missing those fields.
3. Drop `NSA` neighbourhood labels — “Not Specified Area” would distort neighbourhood rankings.
4. Create `total_harm = INJURIES + DEATH` — one severity score for ranking.

This is the **analysis table** both Pandas and DuckDB will use.

In [ ]:
year_col = "OCC_YEAR"
neighbourhood_col = "NEIGHBOURHOOD_158"
event_type_col = "EVENT_TYPE"
injury_col = "INJURIES"
death_col = "DEATH"

clean_df = df.copy()

clean_df[year_col] = pd.to_numeric(clean_df[year_col], errors="coerce")
clean_df[injury_col] = pd.to_numeric(clean_df[injury_col], errors="coerce")
clean_df[death_col] = pd.to_numeric(clean_df[death_col], errors="coerce")

before = len(clean_df)
clean_df = clean_df.dropna(subset=[year_col, injury_col, death_col, neighbourhood_col])
clean_df = clean_df[clean_df[neighbourhood_col] != "NSA"]
clean_df["total_harm"] = clean_df[injury_col] + clean_df[death_col]
clean_df[year_col] = clean_df[year_col].astype(int)

print(f"Rows before cleaning: {before}")
print(f"Rows after cleaning:  {len(clean_df)}")
print(f"Rows removed:         {before - len(clean_df)}")
clean_df[[year_col, neighbourhood_col, event_type_col, injury_col, death_col, "total_harm"]].head()

## 4. Headline descriptive statistics

These totals become your **resume / LinkedIn bullets**. Run them first so every later claim traces back here.

In [ ]:
print("Total incidents:", len(clean_df))
print("Total injuries:", int(clean_df[injury_col].sum()))
print("Total deaths:", int(clean_df[death_col].sum()))
print("Total harm (injuries + deaths):", int(clean_df["total_harm"].sum()))
print()
clean_df[[injury_col, death_col, "total_harm"]].describe()

## 5. Pandas neighbourhood aggregation & three Top-10 rankings

`groupby` + `agg` is the Pandas equivalent of SQL `GROUP BY`.

We rank three ways on purpose:
1. **Incident volume** — how often something was reported
2. **Total harm** — injuries + deaths summed
3. **Harm per incident** — severity relative to volume (small-N places can jump)

**Interview tip:** Explaining *why* you built three rankings (metric sensitivity) is stronger than only showing one chart.

In [ ]:
neighbourhood_summary = (
    clean_df.groupby(neighbourhood_col, as_index=False)
    .agg(
        total_incidents=(neighbourhood_col, "count"),
        total_injuries=(injury_col, "sum"),
        total_deaths=(death_col, "sum"),
        total_harm=("total_harm", "sum"),
    )
)
neighbourhood_summary["harm_per_incident"] = (
    neighbourhood_summary["total_harm"] / neighbourhood_summary["total_incidents"]
)

top_incidents = neighbourhood_summary.sort_values("total_incidents", ascending=False).head(10)
top_harm = neighbourhood_summary.sort_values("total_harm", ascending=False).head(10)
top_harm_rate = neighbourhood_summary.sort_values("harm_per_incident", ascending=False).head(10)

print("=== Top 10 by incidents (Pandas) ===")
display(top_incidents)
print("=== Top 10 by total harm (Pandas) ===")
display(top_harm)
print("=== Top 10 by harm per incident (Pandas) ===")
display(top_harm_rate)

## 6. Visualize Top-10 lists (save PNG charts)

Horizontal bars read well for long neighbourhood names.
`tight_layout()` reduces clipped labels; `savefig` writes portfolio assets into `/charts`.

In [ ]:
def save_barh(frame, value_col, title, xlabel, filename, color="#2c5f7c"):
    plot_df = frame.sort_values(value_col, ascending=True)  # smallest at bottom
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(plot_df[neighbourhood_col], plot_df[value_col], color=color)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Neighbourhood")
    ax.set_title(title)
    fig.tight_layout()
    out = CHARTS / filename
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved:", out)


save_barh(
    top_incidents,
    "total_incidents",
    "Top 10 Neighbourhoods by Firearm-Related Incidents",
    "Total incidents",
    "top10_incidents.png",
)
save_barh(
    top_harm,
    "total_harm",
    "Top 10 Neighbourhoods by Total Harm (Injuries + Deaths)",
    "Total harm",
    "top10_harm.png",
    color="#8b3a3a",
)
save_barh(
    top_harm_rate,
    "harm_per_incident",
    "Top 10 Neighbourhoods by Harm per Incident",
    "Harm per incident",
    "top10_harm_rate.png",
    color="#5a6b3f",
)

## 7. Do volume and harm move together?

Pearson correlation on neighbourhood totals answers: when incidents rise, does harm tend to rise too?
Overlap of the two Top-10 sets answers: are the “busiest” places also the “most harmed”?

In [ ]:
correlation = neighbourhood_summary["total_incidents"].corr(
    neighbourhood_summary["total_harm"]
)
print("Correlation (incidents vs total harm):", round(correlation, 3))

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(
    neighbourhood_summary["total_incidents"],
    neighbourhood_summary["total_harm"],
    alpha=0.7,
    color="#2c5f7c",
)
ax.set_xlabel("Total incidents")
ax.set_ylabel("Total harm (injuries + deaths)")
ax.set_title("Neighbourhood Incidents vs Total Harm")
fig.tight_layout()
fig.savefig(CHARTS / "incidents_vs_harm_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", CHARTS / "incidents_vs_harm_scatter.png")

top_incident_names = set(top_incidents[neighbourhood_col])
top_harm_names = set(top_harm[neighbourhood_col])
overlap = sorted(top_incident_names & top_harm_names)

print()
print("Neighbourhoods in BOTH Top 10 lists:", len(overlap))
for name in overlap:
    print("-", name)

## 8. Rank differences (frequency vs severity vs rate)

Ranks make “who moved” readable. A place can be mid-pack on volume but high on harm-per-incident — useful when discussing how metrics frame risk.

In [ ]:
rank_df = neighbourhood_summary.copy()
rank_df["incident_rank"] = rank_df["total_incidents"].rank(ascending=False, method="min")
rank_df["harm_rank"] = rank_df["total_harm"].rank(ascending=False, method="min")
rank_df["harm_rate_rank"] = rank_df["harm_per_incident"].rank(ascending=False, method="min")
rank_df["rank_gap_incident_vs_harm"] = rank_df["incident_rank"] - rank_df["harm_rank"]

print("Largest gaps: high harm relative to incident rank (negative gap = more harmful than frequent)")
display(
    rank_df.sort_values("rank_gap_incident_vs_harm")[
        [
            neighbourhood_col,
            "total_incidents",
            "total_harm",
            "harm_per_incident",
            "incident_rank",
            "harm_rank",
            "rank_gap_incident_vs_harm",
        ]
    ].head(10)
)

## 9. Trends over time

Yearly aggregates show whether incident volume and harm track across years — context for neighbourhood rankings (which are pooled across the full period).

In [ ]:
yearly_summary = (
    clean_df.groupby(year_col, as_index=False)
    .agg(
        total_incidents=(year_col, "count"),
        total_injuries=(injury_col, "sum"),
        total_deaths=(death_col, "sum"),
        total_harm=("total_harm", "sum"),
    )
    .sort_values(year_col)
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(yearly_summary[year_col], yearly_summary["total_incidents"], marker="o", label="Incidents")
ax.set_xlabel("Year")
ax.set_ylabel("Total incidents")
ax.set_title("Firearm-Related Incidents by Year")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(CHARTS / "incidents_by_year.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(
    yearly_summary[year_col],
    yearly_summary["total_harm"],
    marker="o",
    color="#8b3a3a",
    label="Harm",
)
ax.set_xlabel("Year")
ax.set_ylabel("Total harm (injuries + deaths)")
ax.set_title("Harm Severity by Year")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(CHARTS / "harm_by_year.png", dpi=150, bbox_inches="tight")
plt.show()

print(
    "Saved yearly charts. Year range:",
    yearly_summary[year_col].min(),
    "→",
    yearly_summary[year_col].max(),
)
yearly_summary.head()

## 10. Event type breakdown

`EVENT_TYPE` splits **Shootings** from **Firearm Discharges**.
This matters because harm may concentrate in one type — always check before interpreting citywide totals.

In [ ]:
event_summary = (
    clean_df.groupby(event_type_col, as_index=False)
    .agg(
        total_incidents=(event_type_col, "count"),
        total_injuries=(injury_col, "sum"),
        total_deaths=(death_col, "sum"),
        total_harm=("total_harm", "sum"),
    )
)

fig, ax = plt.subplots(figsize=(8, 6))
ax.bar(
    event_summary[event_type_col],
    event_summary["total_incidents"],
    color=["#5a6b3f", "#8b3a3a"],
)
ax.set_xlabel("Event type")
ax.set_ylabel("Total incidents")
ax.set_title("Incidents by Event Type")
fig.tight_layout()
fig.savefig(CHARTS / "incidents_by_event_type.png", dpi=150, bbox_inches="tight")
plt.show()

display(event_summary)
print("Saved:", CHARTS / "incidents_by_event_type.png")

## 11. DuckDB + SQL — same cleaned data, same rankings

### Why add SQL?
Resume/interview signal: you can move from a DataFrame into a query engine, write `GROUP BY` / `ORDER BY` / `JOIN`, and **validate** Pandas results.

### What happens here
1. Open an in-memory DuckDB connection
2. Register `clean_df` as a virtual table `incidents`
3. Aggregate with SQL
4. Build top-10 lists
5. `JOIN` the incident top-10 to the harm top-10 to list overlap in SQL

Every query below has a line-by-line walkthrough in the markdown after the code.

In [ ]:
# 1) Connect to an in-memory DuckDB database (nothing written to disk unless we choose to)
con = duckdb.connect(database=":memory:")

# 2) Register the cleaned Pandas DataFrame as a queryable table named "incidents"
con.register("incidents", clean_df)

print("DuckDB version:", duckdb.__version__)
print("Registered table 'incidents' with", len(clean_df), "rows")
print("Sample columns:", list(clean_df.columns[:8]), "...")

### SQL Query A — neighbourhood summary (`GROUP BY`)

```sql
SELECT
  NEIGHBOURHOOD_158 AS neighbourhood,  -- rename for clearer output
  COUNT(*) AS total_incidents,         -- one row = one incident
  SUM(INJURIES) AS total_injuries,
  SUM(DEATH) AS total_deaths,
  SUM(total_harm) AS total_harm,       -- uses the engineered column from Pandas
  SUM(total_harm) * 1.0 / COUNT(*) AS harm_per_incident
FROM incidents
GROUP BY NEIGHBOURHOOD_158             -- collapse rows to one per neighbourhood
ORDER BY total_incidents DESC;         -- busiest first (full table, not just top 10)
```

**Line-by-line**
- `SELECT` — choose output columns
- `AS neighbourhood` — alias (like renaming a Series)
- `COUNT(*)` — number of incident rows in the group
- `SUM(...)` — add injuries / deaths / harm inside each neighbourhood
- `* 1.0 / COUNT(*)` — force float division for the rate
- `FROM incidents` — the registered cleaned table
- `GROUP BY NEIGHBOURHOOD_158` — required whenever you aggregate
- `ORDER BY total_incidents DESC` — highest volume first

In [ ]:
sql_neighbourhood_summary = con.execute('''
SELECT
  NEIGHBOURHOOD_158 AS neighbourhood,
  COUNT(*) AS total_incidents,
  SUM(INJURIES) AS total_injuries,
  SUM(DEATH) AS total_deaths,
  SUM(total_harm) AS total_harm,
  SUM(total_harm) * 1.0 / COUNT(*) AS harm_per_incident
FROM incidents
GROUP BY NEIGHBOURHOOD_158
ORDER BY total_incidents DESC
''').df()

print("SQL neighbourhood summary (first 5 rows):")
display(sql_neighbourhood_summary.head())

### SQL Query B — Top 10 by incidents (`ORDER BY` + `LIMIT`)

```sql
SELECT ...
FROM incidents
GROUP BY NEIGHBOURHOOD_158
ORDER BY total_incidents DESC   -- sort high → low
LIMIT 10;                       -- keep only the first 10 rows after sorting
```

`LIMIT` is SQL’s version of Pandas `.head(10)` after a sort.

In [ ]:
sql_top_incidents = con.execute('''
SELECT
  NEIGHBOURHOOD_158 AS neighbourhood,
  COUNT(*) AS total_incidents,
  SUM(total_harm) AS total_harm,
  SUM(total_harm) * 1.0 / COUNT(*) AS harm_per_incident
FROM incidents
GROUP BY NEIGHBOURHOOD_158
ORDER BY total_incidents DESC
LIMIT 10
''').df()

print("=== Top 10 by incidents (SQL / DuckDB) ===")
display(sql_top_incidents)

### SQL Query C — Top 10 by total harm

In [ ]:
sql_top_harm = con.execute('''
SELECT
  NEIGHBOURHOOD_158 AS neighbourhood,
  COUNT(*) AS total_incidents,
  SUM(total_harm) AS total_harm,
  SUM(total_harm) * 1.0 / COUNT(*) AS harm_per_incident
FROM incidents
GROUP BY NEIGHBOURHOOD_158
ORDER BY total_harm DESC
LIMIT 10
''').df()

print("=== Top 10 by total harm (SQL / DuckDB) ===")
display(sql_top_harm)

### SQL Query D — Top 10 by harm per incident

In [ ]:
sql_top_harm_rate = con.execute('''
SELECT
  NEIGHBOURHOOD_158 AS neighbourhood,
  COUNT(*) AS total_incidents,
  SUM(total_harm) AS total_harm,
  SUM(total_harm) * 1.0 / COUNT(*) AS harm_per_incident
FROM incidents
GROUP BY NEIGHBOURHOOD_158
ORDER BY harm_per_incident DESC
LIMIT 10
''').df()

print("=== Top 10 by harm per incident (SQL / DuckDB) ===")
display(sql_top_harm_rate)

### SQL Query E — `JOIN` the two Top-10 lists (overlap)

We materialize two small result sets, then **inner join** on neighbourhood name.
Only places that appear in **both** top-10 lists survive the join — same idea as a set intersection in Python.

```sql
WITH top_inc AS ( ... ORDER BY total_incidents DESC LIMIT 10 ),
     top_hrm AS ( ... ORDER BY total_harm DESC LIMIT 10 )
SELECT
  i.neighbourhood,
  i.total_incidents,
  h.total_harm
FROM top_inc AS i
INNER JOIN top_hrm AS h
  ON i.neighbourhood = h.neighbourhood   -- match keys across the two lists
ORDER BY i.total_incidents DESC;
```

**Line-by-line**
- `WITH ... AS` — Common Table Expression (named subquery you can reuse)
- `top_inc` / `top_hrm` — the two ranking lists
- `INNER JOIN` — keep rows only when the neighbourhood exists in both
- `ON i.neighbourhood = h.neighbourhood` — join key
- `ORDER BY` — present overlap sorted by volume

In [ ]:
sql_overlap = con.execute('''
WITH top_inc AS (
  SELECT
    NEIGHBOURHOOD_158 AS neighbourhood,
    COUNT(*) AS total_incidents
  FROM incidents
  GROUP BY NEIGHBOURHOOD_158
  ORDER BY total_incidents DESC
  LIMIT 10
),
top_hrm AS (
  SELECT
    NEIGHBOURHOOD_158 AS neighbourhood,
    SUM(total_harm) AS total_harm
  FROM incidents
  GROUP BY NEIGHBOURHOOD_158
  ORDER BY total_harm DESC
  LIMIT 10
)
SELECT
  i.neighbourhood,
  i.total_incidents,
  h.total_harm
FROM top_inc AS i
INNER JOIN top_hrm AS h
  ON i.neighbourhood = h.neighbourhood
ORDER BY i.total_incidents DESC
''').df()

print("=== Overlap of Top-10 lists via SQL JOIN ===")
display(sql_overlap)
print("Overlap count (SQL):", len(sql_overlap))

### Validate: Pandas vs SQL must agree

If these checks fail, fix the query — don’t invent a story. Matching tools is a strong interview talking point (“I cross-validated aggregations”).

In [ ]:
# Align column names for comparison
pandas_inc = top_incidents[[neighbourhood_col, "total_incidents", "total_harm"]].reset_index(drop=True)
sql_inc = sql_top_incidents.rename(columns={"neighbourhood": neighbourhood_col}).reset_index(drop=True)

names_match = list(pandas_inc[neighbourhood_col]) == list(sql_inc[neighbourhood_col])
counts_match = list(pandas_inc["total_incidents"]) == list(sql_inc["total_incidents"])

print("Top-10 neighbourhood names match (Pandas vs SQL):", names_match)
print("Top-10 incident counts match (Pandas vs SQL):", counts_match)
print("Overlap count Pandas:", len(overlap), "| SQL:", len(sql_overlap))

assert names_match and counts_match, "Pandas and SQL top-10 by incidents disagree — investigate before reporting."
print("Validation passed.")

## 12. Export summary tables

CSV outputs are regenerable (gitignored). Charts in `/charts` are kept for the README / LinkedIn.

In [ ]:
neighbourhood_summary.to_csv("neighbourhood_summary.csv", index=False)
yearly_summary.to_csv("yearly_summary.csv", index=False)
event_summary.to_csv("event_summary.csv", index=False)
sql_neighbourhood_summary.to_csv("sql_neighbourhood_summary.csv", index=False)

print("Wrote neighbourhood_summary.csv, yearly_summary.csv, event_summary.csv, sql_neighbourhood_summary.csv")
print("Charts in:", sorted(p.name for p in CHARTS.glob("*.png")))

## 13. Findings & limitations (use only numbers from this run)

### Key findings (from this notebook’s outputs)
1. **Scale after cleaning:** 6,759 incidents; 3,114 injuries; 788 deaths; **3,902** total harm.
2. **Volume ↔ harm:** neighbourhood incident counts and total harm correlate at **0.968**; **6** neighbourhoods sit in both Top-10 lists (Black Creek, Glenfield-Jane Heights, West Humber-Clairville, Mount Olive-Silverstone-Jamestown, Yorkdale-Glen Park, York University Heights). Glenfield-Jane Heights leads volume at **347** incidents.
3. **Metric sensitivity:** harm-per-incident ranks a different set (e.g. Playter Estates-Danforth at **1.83** harm/incident on **12** incidents). Event split: **3,600** firearm discharges (0 recorded injuries/deaths here) vs **3,159** shootings (all of the injuries/deaths above).

### Limitations (say these in interviews)
- Open data is **administrative** — reporting practices and definitions can change over years.
- Dropping `NSA` improves neighbourhood ranking quality but removes incidents with unclear geography.
- Harm-per-incident on **small counts** is unstable (one multi-victim event swings the rate).
- Correlation ≠ causation; neighbourhood rankings are descriptive, not causal.
- `NEIGHBOURHOOD_158` boundaries are one geography choice among others (e.g. 140).

### How to talk about SQL here
You didn’t just “use Pandas.” You loaded the same cleaned table into DuckDB, rebuilt Top-10 rankings with `GROUP BY` / `ORDER BY` / `LIMIT`, and used a `JOIN` of two CTEs to compute Top-10 overlap — then asserted Pandas and SQL agree.